# Session VII: Command Line Tools and Automation

Use Python to automate tasks and interact through the command line, including parsing arguments and running scripts from the terminal.

## Exercise 1: CLI Greeter and Calculator
**Objective:** Practice basic command-line parsing using a list that represents `sys.argv`.

Instead of reading real terminal input in the notebook, you will write a function that receives an argument list like:

```python
["greet", "Alice"]
["add", "3", "4"]
```

**Tasks:**
1. `run_command(argv)` should support two commands:
   - `greet <name>` → return `"Hello, <name>!"`
   - `add <x> <y>` → return the numeric sum as a string
2. Return a helpful usage message for missing or invalid input.
3. Use `try/except` to handle non-numeric values in `add`.


In [1]:
USAGE = (
    "Usage:\n"
    "  greet <name>\n"
    "  add <x> <y>"
)


def run_command(argv):
    """Parse a simple command list and return a response string."""
    if not argv:
        return None

    command, *args = argv

    if command == "greet":
        if len(args) != 1:
            return "Usage: greet <name>"
        return f"Hello, {args[0]}!"

    if command == "add":
        if len(args) != 2:
            return "Usage: add <x> <y>"
        try:
            x = float(args[0])
            y = float(args[1])
        except ValueError:
            return f"Error: 'add' needs two numbers, got {args[0]!r} and {args[1]!r}."
        return str(x + y)

    return f"Unknown command: {command!r}\n{USAGE}"


# --- TEST ---
print(run_command(["greet", "Alice"]))
# Expected: Hello, Alice!

print(run_command(["add", "3", "4"]))
# Expected: 7.0

print(run_command(["add", "three", "4"]))
# Expected: Error message about numeric input


Hello, Alice!
7.0
Error: 'add' needs two numbers, got 'three' and '4'.


---
## Exercise 2: Expense Lister with `argparse`
**Objective:** Build a real argument parser using optional flags.

You are given a list of expense records. Build a command-line style filter that can:
- show all expenses
- filter by category
- filter by a minimum amount

**Tasks:**
1. `build_parser()` → Create an `ArgumentParser` with:
   - `--category`
   - `--min` (float, default `0`)
2. `filter_expenses(expenses, category=None, min_amount=0)` → Return only matching expenses.
3. `format_expenses(expenses)` → Return a formatted multi-line string.


In [2]:
import argparse

expenses = [
    {"date": "2026-10-01", "category": "Food", "amount": 18.50, "note": "Lunch"},
    {"date": "2026-10-02", "category": "Transport", "amount": 2.80, "note": "Metro"},
    {"date": "2026-10-02", "category": "Food", "amount": 34.20, "note": "Groceries"},
    {"date": "2026-10-03", "category": "Books", "amount": 45.00, "note": "Python book"},
]

def build_parser():
    """Return an argparse parser for expense filtering."""
    # TODO: Create ArgumentParser
    parser = argparse.ArgumentParser(description="Expense Filtering")

    # TODO: Add optional --category argument
    parser.add_argument("--category", help="Filter by Category")

    # TODO: Add optional --min argument (type=float, default=0)
    parser.add_argument("--min", type=float, default=0, help="Minimum Amount")

    return parser

def filter_expenses(expenses, category=None, min_amount=0):
    """Return filtered expenses matching the given criteria."""
    # TODO: Loop through expenses and keep only matching rows
    filtered = []
    for expense in expenses:
        if ((category is None or expense["category"].lower() == category.lower())
                and expense["amount"] >= min_amount):
            filtered.append(expense)
    return filtered

def format_expenses(expenses):
    """Return a formatted text report for a list of expenses."""
    # TODO: Handle empty list
    if not expenses:
        return "No filtered expenses."

    # TODO: Build a list of output lines
    lines = [f"{'Date':<12}{'Category':<8}{'Amount':>10}  Note"]
    lines.append("-" * 44)
    for expense in expenses:
        lines.append(
            f"{expense['date']:<12}{expense['category']:<8}"
            f"{expense['amount']:>10.2f}  {expense['note']}"
        )
    total = sum(expense["amount"] for expense in expenses)
    lines.append("-" * 44)
    lines.append(f"{'Total':<20}{total:>10.2f}")

    # TODO: Return one multi-line string
    return "\n".join(lines)


# --- TEST ---
parser = build_parser()
args = parser.parse_args(["--category", "Food", "--min", "20"])
filtered = filter_expenses(expenses, args.category, args.min)
print(format_expenses(filtered))


Date        Category    Amount  Note
--------------------------------------------
2026-10-02  Food         34.20  Groceries
--------------------------------------------
Total                    34.20


---
## Exercise 3: Budget Backup Script
**Objective:** Automate data backups using `pathlib`, timestamps, and file copying.

Your script should validate that the source file exists, create a `backups` folder if needed, and make a timestamped copy.

**Tasks:**
1. `make_backup_name(filename)` → Return a new filename like `budget_20261005_224700.json`
2. `backup_file(source_path, backup_dir="backups")` → Copy the source file into the backup folder.
3. Return the destination path as a string, or `None` if the source file is missing.

**First, run the setup cell below to create a sample file.**


In [3]:
from pathlib import Path
import json

sample_budget = [
    {"category": "Food", "amount": 20.5},
    {"category": "Transport", "amount": 15.0},
]

with open("budget_data.json", "w") as file:
    json.dump(sample_budget, file, indent=2)

print("Created budget_data.json")


Created budget_data.json


In [4]:
from pathlib import Path
from datetime import datetime
import shutil

def make_backup_name(filename):
    """Return a timestamped backup filename."""
    file_path = Path(filename)

    # TODO: Use Path(filename).stem and .suffix
    suffix = file_path.suffix
    origin_filename = file_path.stem

    # TODO: Add a timestamp with datetime.now().strftime(...)
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    return f"{origin_filename}_{timestamp}{suffix}"

def backup_file(source_path, backup_dir="backups"):
    """Copy a file into backup_dir with a timestamped name."""
    # TODO: Convert to Path objects
    source = Path(source_path)
    backup = Path(backup_dir)

    # TODO: Check whether source exists
    if not source.is_file():
        print(f"Error: source file {str(source)!r} not found.")
        return None

    # TODO: Create backup folder if needed
    backup.mkdir(parents=True, exist_ok=True)

    # TODO: Copy file with shutil.copy2
    destination = backup / make_backup_name(source.name)
    shutil.copy2(source, destination)

    # TODO: Return destination path as a string
    return str(destination)


# --- TEST ---
print(make_backup_name("budget_data.json"))
# Expected: budget_data_YYYYMMDD_HHMMSS.json

result = backup_file("budget_data.json")
print("Backup created:", result)
# Expected: backups/budget_data_YYYYMMDD_HHMMSS.json

print(backup_file("missing_file.json"))
# Expected: error message, then None


budget_data_20261112_180632.json
Backup created: backups/budget_data_20261112_180632.json
Error: source file 'missing_file.json' not found.
None


---
## Exercise 4: CSV Expense Summary CLI
**Objective:** Load a CSV file, validate rows, and print category totals.

**Tasks:**
1. `load_expenses_csv(filename)` → Use `csv.DictReader` to load rows.
2. Skip malformed rows where:
   - `category` is empty
   - `amount` is missing or non-numeric
3. `summarize_by_category(expenses)` → Return a dictionary of totals.
4. `print_summary(summary)` → Print a formatted report.

**Run the setup cell below first.**


In [5]:
import csv

rows = [
    ["date", "category", "amount", "note"],
    ["2026-10-01", "Food", "18.50", "Lunch"],
    ["2026-10-02", "Transport", "2.80", "Metro"],
    ["2026-10-02", "", "12.00", "Broken row"],
    ["2026-10-03", "Books", "forty", "Invalid amount"],
    ["2026-10-04", "Food", "21.30", "Dinner"],
]

with open("expenses_cli.csv", "w", newline="") as file:
    writer = csv.writer(file)
    writer.writerows(rows)

print("Created expenses_cli.csv")


Created expenses_cli.csv


In [6]:
import csv

def load_expenses_csv(filename):
    """Load valid expense rows from a CSV file."""
    expenses = []

    # TODO: Use csv.DictReader
    with open(filename, newline="") as file:
        reader = csv.DictReader(file)
        for line_number, row in enumerate(reader, start=2):
            # TODO: Validate category and amount
            category = (row.get("category") or "").strip()
            amount = (row.get("amount") or "").strip()

            if not category:
                print(f"Skipping line {line_number}: missing category.")
                continue

            # TODO: Convert amount to float
            # TODO: Skip bad rows safely
            try:
                amount = float(amount)
            except ValueError:
                print(f"Skipping line {line_number}: invalid amount {amount!r}.")
                continue

            row["category"] = category
            row["amount"] = amount
            expenses.append(row)

    return expenses

def summarize_by_category(expenses):
    """Return {category: total_amount}."""
    # TODO: Accumulate totals in a dictionary
    summary = {}
    for expense in expenses:
        category = expense["category"]
        summary[category] = summary.get(category, 0) + expense["amount"]
    return summary

def print_summary(summary):
    """Print a formatted category summary."""
    # TODO: Handle empty summary
    if not summary:
        print("No expenses to summarize.")
        return

    # TODO: Print aligned output
    print(f"{'Category':<12}{'Total':>10}")
    print("-" * 22)
    for category, total in summary.items():
        print(f"{category:<12}{total:>10.2f}")
    print("-" * 22)
    print(f"{'Total':<12}{sum(summary.values()):>10.2f}")


# --- TEST ---
expenses = load_expenses_csv("expenses_cli.csv")
summary = summarize_by_category(expenses)
print_summary(summary)


Skipping line 4: missing category.
Skipping line 5: invalid amount 'forty'.
Category         Total
----------------------
Food             39.80
Transport         2.80
----------------------
Total            42.60


---
## Exercise 5: Scheduling Helper
**Objective:** Generate schedule strings for cron and Windows Task Scheduler.

You are **not** actually scheduling tasks in this notebook. Instead, you will generate the correct command strings.

**Tasks:**
1. `build_cron_schedule(minute, hour, script_path)` → Return a cron line like:
   `30 18 * * * python /path/to/script.py`
2. `build_task_scheduler_command(time_str, script_path)` → Return a Windows-style command string.
3. Validate the time values and raise or handle `ValueError` for invalid input.


In [7]:
def build_cron_schedule(minute, hour, script_path):
    """Return a cron schedule line for a daily script."""
    # TODO: Validate minute (0-59) and hour (0-23)
    if not 0 <= minute <= 59:
        raise ValueError(f"minute must be between 0 and 59, got {minute}")
    if not 0 <= hour <= 23:
        raise ValueError(f"hour must be between 0 and 23, got {hour}")

    # TODO: Return the cron line as a string
    return f"{minute} {hour} * * * python {script_path}"

def build_task_scheduler_command(time_str, script_path):
    """Return a Windows schtasks command string."""
    # TODO: Validate HH:MM format
    parts = time_str.split(":")
    if len(parts) != 2 or not all(part.isdigit() and len(part) == 2 for part in parts):
        raise ValueError(f"time must be in HH:MM format, got {time_str!r}")

    hour, minute = int(parts[0]), int(parts[1])
    if hour > 23 or minute > 59:
        raise ValueError(f"invalid time {time_str!r}")

    # TODO: Return a command like:
    # schtasks /Create /SC DAILY /TN BudgetBackup /TR "python script.py" /ST 18:30
    return f'schtasks /Create /SC DAILY /TN BudgetBackup /TR "python {script_path}" /ST {time_str}'


# --- TEST ---
print(build_cron_schedule(30, 18, "/home/user/backup.py"))
print(build_task_scheduler_command("18:30", "C:/scripts/backup.py"))

try:
    build_cron_schedule(75, 18, "/home/user/backup.py")
except ValueError as error:
    print("Error:", error)

try:
    build_task_scheduler_command("6pm", "C:/scripts/backup.py")
except ValueError as error:
    print("Error:", error)


30 18 * * * python /home/user/backup.py
schtasks /Create /SC DAILY /TN BudgetBackup /TR "python C:/scripts/backup.py" /ST 18:30
Error: minute must be between 0 and 59, got 75
Error: time must be in HH:MM format, got '6pm'


---
## Exercise 6: Automation Suite
**Objective:** Combine parsing, loading JSON, listing expenses, backups, and summaries into one mini CLI application.

Build a tool with these subcommands:
- `list` → print all expenses
- `summary` → show totals by category
- `backup` → make a timestamped backup of the JSON file

**Tasks:**
1. `load_json_expenses(filename)` → Safely load a JSON list of expense dictionaries.
2. `build_suite_parser()` → Create an `argparse` parser with subcommands:
   - `list`
   - `summary`
   - `backup`
   Also add a common `filename` argument.
3. `run_suite(argv=None)` → Parse the arguments and perform the correct action.
4. Reuse functions from earlier exercises when possible.

**Run the setup cell below first.**


In [8]:
import json

suite_data = [
    {"date": "2026-10-01", "category": "Food", "amount": 18.50, "note": "Lunch"},
    {"date": "2026-10-02", "category": "Transport", "amount": 2.80, "note": "Metro"},
    {"date": "2026-10-03", "category": "Food", "amount": 21.30, "note": "Dinner"},
    {"date": "2026-10-04", "category": "Books", "amount": 45.00, "note": "Python book"},
]

with open("expenses_suite.json", "w") as file:
    json.dump(suite_data, file, indent=2)

print("Created expenses_suite.json")


Created expenses_suite.json


In [9]:
import argparse
import json

def load_json_expenses(filename):
    """Safely load and validate a JSON expense list."""
    # TODO: Handle FileNotFoundError and json.JSONDecodeError
    try:
        with open(filename) as file:
            data = json.load(file)
    except FileNotFoundError:
        print(f"Error: file {filename!r} not found.")
        return []
    except json.JSONDecodeError:
        print(f"Error: {filename!r} is not valid JSON.")
        return []

    # TODO: Ensure the top-level object is a list
    if not isinstance(data, list):
        print(f"Error: expected a list of expenses in {filename!r}.")
        # TODO: Return [] on error
        return []

    return data

def build_suite_parser():
    """Return an argparse parser with subcommands."""
    # TODO: Create parser
    parser = argparse.ArgumentParser(description="Expense Automation Suite")

    # TODO: Add a required filename argument
    parser.add_argument("filename", help="Path to the JSON expense file")

    # TODO: Add subcommands: list, summary, backup
    subparsers = parser.add_subparsers(dest="command", required=True)
    subparsers.add_parser("list", help="Show all expenses")
    subparsers.add_parser("summary", help="Show totals by category")
    subparsers.add_parser("backup", help="Make a timestamped backup of the file")

    return parser

def run_suite(argv=None):
    """Run the automation suite and return a message or report."""
    # TODO: Parse args
    parser = build_suite_parser()
    try:
        args = parser.parse_args(argv)
    except SystemExit:
        return parser.format_usage()

    # TODO: Route to list / summary / backup
    if args.command == "backup":
        result = backup_file(args.filename)
        if result is None:
            return "Backup failed."
        return f"Backup created: {result}"

    # TODO: Load data when needed
    expenses = load_json_expenses(args.filename)

    if args.command == "list":
        return format_expenses(expenses)

    if args.command == "summary":
        summary = summarize_by_category(expenses)
        if not summary:
            return "No expenses to summarize."
        lines = [f"{'Category':<12}{'Total':>10}", "-" * 22]
        for category, total in summary.items():
            lines.append(f"{category:<12}{total:>10.2f}")
        return "\n".join(lines)


# --- TEST ---
print(run_suite(["expenses_suite.json", "list"]))
print()
print(run_suite(["expenses_suite.json", "summary"]))
print()
print(run_suite(["expenses_suite.json", "backup"]))


Date        Category    Amount  Note
--------------------------------------------
2026-10-01  Food         18.50  Lunch
2026-10-02  Transport      2.80  Metro
2026-10-03  Food         21.30  Dinner
2026-10-04  Books        45.00  Python book
--------------------------------------------
Total                    87.60

Category         Total
----------------------
Food             39.80
Transport         2.80
Books            45.00

Backup created: backups/expenses_suite_20261112_180632.json
